# Earthquake effect on house prices

In [14]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms

from read_cbs_data import *


In [15]:
CONFOUNDER_FOLDER_PATH = "./data/confounders/"

In [16]:
gdf_gemeenten = concatenate_cbs_gebieden(list(range(2010, 2026)), "gemeente_gegeneraliseerd")
gdf_provincies = concatenate_cbs_gebieden(list(range(2010, 2026)), "provincie_gegeneraliseerd")

gdf_joined = (
    join_gemeente_with_provincie(gdf_gemeenten, gdf_provincies)
    .rename({'statnaam__gemeente': "Regio's", 'jaar__gemeente': 'Perioden'}, axis = 1)
    )

gdf_joined.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 6059 entries, 0 to 6058
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   Regio's              6059 non-null   object  
 1   geometry             6059 non-null   geometry
 2   Perioden             6059 non-null   int64   
 3   statnaam__provincie  5852 non-null   object  
dtypes: geometry(1), int64(1), object(2)
memory usage: 189.5+ KB


In [17]:
gdf_joined.head()

,Regio's,geometry,Perioden,statnaam__provincie
0,Appingendam,"POLYGON ((254983.406 592487.499, 255044.543 59...",2010,Groningen
1,Bedum,"POLYGON ((235435.98 595111.363, 235464.991 595...",2010,Groningen
2,Bellingwedde,"POLYGON ((276518.07 566843.126, 276821.63 5647...",2010,Groningen
3,Ten Boer,"POLYGON ((245005.54 592656.73, 245209.866 5925...",2010,Groningen
4,Delfzijl,"POLYGON ((263194.613 592290.99, 264285.625 591...",2010,Groningen


In [18]:
df_prijsindex = read_prijsindex_data()

Index(['Gemeentecode', 'Gemeentenaam', 'Index 2020=100',
       '95% betrouwbaarheidsmarge ondergrens',
       '95% betrouwbaarheidsmarge bovengrens',
       'Ontwikkeling t.o.v. voorgaande periode',
       'Ontwikkeling t.o.v. een jaar eerder', 'Jaar'],
      dtype='object')

In [19]:
df_tmp = gdf_joined.merge(df_prijsindex, left_on=["Regio's", 'Perioden'], right_on=['Gemeentenaam', 'Jaar'], how='inner')
df_tmp.shape

(4423, 12)

In [20]:
df_tmp.head()

,Regio's,geometry,Perioden,statnaam__provincie,Gemeentecode,Gemeentenaam,Index 2020=100,95% betrouwbaarheidsmarge ondergrens,95% betrouwbaarheidsmarge bovengrens,Ontwikkeling t.o.v. voorgaande periode,Ontwikkeling t.o.v. een jaar eerder,Jaar
0,Almere,"POLYGON ((144961.673 482396.738, 143843.369 48...",2010,Flevoland,34,Almere,65.0,63.5,66.6,-1.5,-1.3,2010
1,Stadskanaal,"POLYGON ((267908.252 552996.903, 267743.215 55...",2010,Groningen,37,Stadskanaal,76.4,73.6,79.2,-1.2,-3.2,2010
2,Veendam,"POLYGON ((256515.077 572257.451, 258623.14 571...",2010,Groningen,47,Veendam,81.3,78.2,84.5,-1.0,-1.2,2010
3,Zeewolde,"POLYGON ((168731.104 491504.672, 171049.296 49...",2010,Flevoland,50,Zeewolde,81.8,76.9,86.8,-3.4,-5.7,2010
4,Achtkarspelen,"POLYGON ((205932.919 574747.372, 205900.704 57...",2010,Friesland,59,Achtkarspelen,81.3,79.0,83.7,-2.4,-2.4,2010


In [21]:
type(df_tmp)

geopandas.geodataframe.GeoDataFrame

In [22]:
df_tmp.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 4423 entries, 0 to 4422
Data columns (total 12 columns):
 #   Column                                  Non-Null Count  Dtype   
---  ------                                  --------------  -----   
 0   Regio's                                 4423 non-null   object  
 1   geometry                                4423 non-null   geometry
 2   Perioden                                4423 non-null   int64   
 3   statnaam__provincie                     4236 non-null   object  
 4   Gemeentecode                            4423 non-null   object  
 5   Gemeentenaam                            4423 non-null   object  
 6   Index 2020=100                          4423 non-null   float64 
 7   95% betrouwbaarheidsmarge ondergrens    4423 non-null   float64 
 8   95% betrouwbaarheidsmarge bovengrens    4423 non-null   float64 
 9   Ontwikkeling t.o.v. voorgaande periode  4423 non-null   float64 
 10  Ontwikkeling t.o.v. een jaar eerder     

In [26]:
tmp = merge_confounder_data(df_tmp)

Perioden is object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6960 entries, 0 to 6959
Data columns (total 4 columns):
 #   Column                                                               Non-Null Count  Dtype  
---  ------                                                               --------------  -----  
 0   Soort misdrijf                                                       6960 non-null   object 
 1   Perioden                                                             6960 non-null   object 
 2   Regio's                                                              6960 non-null   object 
 3   Geregistreerde misdrijven/Totaal geregistreerde misdrijven (aantal)  5732 non-null   float64
dtypes: float64(1), object(3)
memory usage: 217.6+ KB
None
Perioden is object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9500 entries, 0 to 9499
Data columns (total 13 columns):
 #   Column                                                                                   Non-Null

In [31]:
tmp.to_file("./results/areas_prepped.gpkg", layer="data", driver="GPKG")


In [29]:
tmp.crs

<Projected CRS: EPSG:28992>
Name: Amersfoort / RD New
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: Netherlands - onshore, including Waddenzee, Dutch Wadden Islands and 12-mile offshore coastal zone.
- bounds: (3.2, 50.75, 7.22, 53.7)
Coordinate Operation:
- name: RD New
- method: Oblique Stereographic
Datum: Amersfoort
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

In [32]:
df_aardbevingen = read_aardbevingen_data()
gdf_aardbevingen = transform_aardbevingen_data(df_aardbevingen)

In [33]:
gdf_aardbevingen.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [34]:
gdf_aardbevingen.to_file("./results/aardbevingen.gpkg", layer="data", driver="GPKG")